# Borana Drought Outlook (SPI-based) - real data, honest testing

**Why this notebook exists:** the dashboard's old forecast is a *simulation* (random numbers). The destocking advisor needs a **real** answer to: *"how likely is a drought in Borana during the next 3 months?"*

**What it does, step by step (plain words):**
1. Downloads real rainfall history (1981 to last month) for several points across Borana.
2. Turns rainfall into the **SPI** (Standardized Precipitation Index): "how much drier or wetter than normal is this 3-month period?" Below -1 is a drought.
3. Trains a small, simple model that uses *today's* dryness and the ENSO / Indian Ocean climate signals to give the **chance of No Drought / Mild / Moderate / Severe** for the next 3 months.
4. **Tests it honestly**: trained on old years, checked on later years it never saw (including the 2020-2023 drought), and compared with two simple baselines. If it does not beat them, the notebook says so and uses the simple version.
5. Writes `drought_outlook.json` in exactly the format your advisor reads.

**Honest limits you should repeat in any report:** rainfall in the Horn of Africa is hard to forecast 1-3 months ahead; expect only modest skill. Rainfall here is ERA5 reanalysis (not rain-gauge data). SPI measures rainfall, not forage, soil moisture or animal deaths.

In [ ]:
import json, time, datetime, warnings
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, recall_score
warnings.filterwarnings("ignore")

KEYS = ["No Drought", "Mild", "Moderate", "Severe"]       # same 4 classes the advisor uses
# SPI thresholds for the class of the NEXT 3 MONTHS (a choice, written down so it can be debated):
#   No Drought: SPI >= -0.5 | Mild: -1.0 to -0.5 | Moderate: -1.5 to -1.0 | Severe: below -1.5
def spi_to_class(x):
    x = np.asarray(x, dtype=float)
    return np.select([x >= -0.5, x >= -1.0, x >= -1.5], [0, 1, 2], default=3)

# Points spread over the Borana zone (approximate town locations). Edit freely.
POINTS = {"Yabello": (4.88, 38.08), "Arero": (4.75, 38.83), "Moyale": (3.52, 39.05),
          "Mega": (4.05, 38.30), "Dire": (4.55, 38.25), "Miyo": (3.90, 38.80)}
START_YEAR = 1981
BASELINE = (1981, 2010)                                   # the 'normal' climate SPI is measured against
MARKETS_TO_EXPORT = ["Miyo", "Yabalo", "Nagelle Borana", "Arero", "Moyale Oromiya"]

## 1. Download real data

Three downloads (all free, no account):
- **Rainfall:** daily ERA5 rainfall from the Open-Meteo archive for each Borana point, added up to months and averaged over the points.
- **ENSO (ONI)** and **Indian Ocean Dipole (DMI):** climate signals that influence East African rains. DMI is optional; if its website changes the notebook carries on without it.

If a download fails, wait a few minutes and run the cell again. Results are saved so you do not repeat them.

In [ ]:
def last_complete_month_end():
    first_of_this_month = datetime.date.today().replace(day=1)
    return first_of_this_month - datetime.timedelta(days=1)

END = last_complete_month_end()

def fetch_point(lat, lon, start, end, retries=5):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = dict(latitude=lat, longitude=lon, start_date=start, end_date=end, daily="precipitation_sum", timezone="UTC")
    for i in range(retries):
        r = requests.get(url, params=params, timeout=180)
        if r.status_code == 200:
            d = r.json()["daily"]
            return pd.Series(d["precipitation_sum"], index=pd.to_datetime(d["time"]), dtype=float)
        time.sleep(10 * (i + 1))
    raise RuntimeError("Rainfall download failed: HTTP %s %s" % (r.status_code, r.text[:200]))

monthly = {}
for name, (lat, lon) in POINTS.items():
    s = fetch_point(lat, lon, f"{START_YEAR}-01-01", END.isoformat())
    monthly[name] = s.resample("MS").sum(min_count=20)           # months with too many missing days become NaN
    print(name, "ok", len(s), "days"); time.sleep(3)
rain_points = pd.DataFrame(monthly)
rain_points.to_csv("borana_rain_points_cache.csv")
rain_monthly = rain_points.mean(axis=1).rename("rain_mm")          # Borana-average monthly rainfall
print(rain_monthly.tail(6).round(1))

In [ ]:
# ENSO (ONI): 3-month running sea-surface-temperature anomaly in the Pacific
enso = None
try:
    t = requests.get("https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt", timeout=60).text.strip().splitlines()[1:]
    seas = ["DJF","JFM","FMA","MAM","AMJ","MJJ","JJA","JAS","ASO","SON","OND","NDJ"]
    rows = []
    for line in t:
        p = line.split()
        if len(p) >= 4 and p[0] in seas:
            rows.append((pd.Timestamp(int(p[1]), seas.index(p[0]) + 1, 1), float(p[3])))
    enso = pd.Series(dict(rows)).sort_index().rename("oni")
    print("ONI ok, last value", enso.index[-1].date(), enso.iloc[-1])
except Exception as e:
    print("ONI not available:", e)

# Indian Ocean Dipole (DMI), optional
iod = None
try:
    t = requests.get("https://psl.noaa.gov/gcos_wgsp/Timeseries/Data/dmi.had.long.data", timeout=60).text.splitlines()
    rows = []
    for line in t:
        p = line.split()
        if len(p) == 13 and p[0].isdigit():
            for m in range(12):
                v = float(p[m + 1])
                if v > -99: rows.append((pd.Timestamp(int(p[0]), m + 1, 1), v))
    iod = pd.Series(dict(rows)).sort_index().rename("dmi")
    print("DMI ok, last value", iod.index[-1].date(), iod.iloc[-1])
except Exception as e:
    print("DMI not available (carrying on without it):", e)

## 2. Rainfall -> SPI -> drought classes

**SPI in one sentence:** compare this 3-month rainfall total with all the same 3-month periods in the "normal" years (1981-2010) and say how unusual it is. 0 = normal, -1 = fairly dry, -1.5 or lower = severely dry.

The target the model must predict is the **class of the SPI of the NEXT 3 months** (the same window the advisor cares about).

In [ ]:
def spi_series(monthly_rain, k):
    acc = monthly_rain.rolling(k, min_periods=k).sum()
    out = pd.Series(np.nan, index=acc.index)
    for m in range(1, 13):
        sel = acc[acc.index.month == m].dropna()
        base = sel[(sel.index.year >= BASELINE[0]) & (sel.index.year <= BASELINE[1])]
        pos = base[base > 0]
        if len(pos) < 10:
            continue
        p0 = (base <= 0).mean()
        a, loc, scale = stats.gamma.fit(pos, floc=0)
        cdf = p0 + (1 - p0) * stats.gamma.cdf(sel.clip(lower=0), a, loc=0, scale=scale)
        out.loc[sel.index] = stats.norm.ppf(np.clip(cdf, 1e-6, 1 - 1e-6))
    return out

def build_dataset(rain_monthly, enso, iod):
    r = rain_monthly.copy()
    df = pd.DataFrame({"rain": r})
    for k in (1, 3, 6, 12):
        df[f"spi{k}"] = spi_series(r, k)
    ang = 2 * np.pi * df.index.month / 12
    df["m_sin"], df["m_cos"] = np.sin(ang), np.cos(ang)
    # No peeking at the future: ONI for month t is only known ~2 months later, DMI ~1 month later.
    if enso is not None: df["oni"] = enso.reindex(df.index).shift(2).ffill()
    if iod is not None:  df["dmi"] = iod.reindex(df.index).shift(1).ffill()
    df["target_spi"] = df["spi3"].shift(-3)             # SPI of the 3 months AFTER this month
    df["target"] = np.where(df["target_spi"].notna(), spi_to_class(df["target_spi"].fillna(0)), -1)
    return df

df = build_dataset(rain_monthly, enso, iod)
FEATURES = [c for c in ["spi1", "spi3", "spi6", "spi12", "m_sin", "m_cos", "oni", "dmi"] if c in df.columns]
print("features used:", FEATURES)
print(df.dropna(subset=FEATURES).tail(4).round(2)[FEATURES + ["target_spi"]])

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(df.index, df["spi3"], lw=1)
for y, c in [(-0.5, "gold"), (-1.0, "orange"), (-1.5, "red")]: ax.axhline(y, color=c, ls="--", lw=0.8)
ax.set_title("Borana 3-month SPI (below -1 = drought). Check: do the known droughts (2011, 2016-17, 2020-23) show up?")
plt.tight_layout(); plt.show()
print("Share of months in each class (all years):", dict(zip(KEYS, np.bincount(df['target'][df['target'] >= 0], minlength=4) / (df['target'] >= 0).sum())))

## 3. Train and test honestly (time-based, no shuffling)

- **Train:** 1981-2012 &nbsp; **Check:** 2013-2018 &nbsp; **Final test:** 2019-2025 (includes the 2020-2023 drought the model never saw)
- **Two simple baselines to beat:**
  1. *Climatology* - "just use how often each class happened in training for this calendar month".
  2. *Persistence* - "next 3 months will be like the last 3 months".
- **Scores:** *Brier score* (lower is better, judges the probabilities), *Brier skill score* (above 0 = better than climatology), macro-F1 and recall for Severe drought.

In [ ]:
usable = df.dropna(subset=FEATURES)
usable = usable[usable["target"] >= 0]
tr = usable[usable.index.year <= 2012]
va = usable[(usable.index.year >= 2013) & (usable.index.year <= 2018)]
te = usable[usable.index.year >= 2019]
print("train/check/test months:", len(tr), len(va), len(te))

def onehot(y): return np.eye(4)[np.asarray(y, dtype=int)]
def brier(P, y): return float(np.mean(np.sum((P - onehot(y)) ** 2, axis=1)))

def climatology_probs(train, months):
    glob = np.bincount(train["target"], minlength=4) / len(train)
    out = []
    for m in months:
        sub = train[train.index.month == m]["target"]
        counts = np.bincount(sub, minlength=4) + 5 * glob          # smoothed toward the overall rate
        out.append(counts / counts.sum())
    return np.array(out)

def persistence_probs(spi3_now):
    cls = spi_to_class(spi3_now)
    P = np.full((len(cls), 4), 0.1); P[np.arange(len(cls)), cls] = 0.7
    return P

def fit_model(train):
    m = make_pipeline(StandardScaler(), LogisticRegression(C=0.3, max_iter=2000))
    m.fit(train[FEATURES], train["target"])
    return m

def full_probs(model, X):
    P = np.zeros((len(X), 4)); P[:, model.classes_] = model.predict_proba(X); return P

def evaluate(name, P, y):
    pred = P.argmax(axis=1)
    sev_recall = recall_score(y == 3, pred == 3) if (y == 3).any() else float("nan")
    return {"set": name, "n": len(y), "brier": brier(P, y), "macro_F1": f1_score(y, pred, average="macro"), "severe_recall": sev_recall}

model = fit_model(tr)
rows, bss = [], {}
for nm, part in [("check 2013-2018", va), ("test 2019-2025", te)]:
    y = part["target"].values
    P_clim = climatology_probs(tr, part.index.month)
    P_pers = persistence_probs(part["spi3"].values)
    P_mod = full_probs(model, part[FEATURES])
    for label, P in [("climatology", P_clim), ("persistence", P_pers), ("model", P_mod)]:
        r = evaluate(nm, P, y); r["method"] = label; rows.append(r)
    bss[nm] = 1 - brier(P_mod, y) / brier(P_clim, y)
res = pd.DataFrame(rows)[["set", "method", "n", "brier", "macro_F1", "severe_recall"]].round(3)
print(res.to_string(index=False))
print("\nBrier skill score of the model vs climatology (above 0 = useful):", {k: round(v, 3) for k, v in bss.items()})

In [ ]:
# Calibration: when the model says 'chance of Moderate or Severe = X', how often did it really happen? (check + test years together)
part = pd.concat([va, te]); P = full_probs(model, part[FEATURES]); y = part["target"].values
p_bad = P[:, 2] + P[:, 3]; happened = (y >= 2).astype(float)
bins = [0, 0.1, 0.2, 0.3, 0.5, 1.01]
cal = pd.DataFrame({"bin": pd.cut(p_bad, bins, right=False), "pred": p_bad, "obs": happened}).groupby("bin").agg(n=("obs", "size"), mean_predicted=("pred", "mean"), actually_happened=("obs", "mean")).round(3)
print(cal)

## 4. Decide what the advisor may use

Rule written down in advance: the model is used only if it beats climatology (skill above 0) on **both** the check years and the test years. Otherwise the advisor gets the plain climatology probabilities, labelled as such. No bluffing.

In [ ]:
model_ok = all(v > 0 for v in bss.values())
METHOD = "logistic model (ENSO/IOD + SPI persistence)" if model_ok else "climatology only (model did not beat it)"
print("Model beats climatology on both periods?", model_ok, "->", METHOD)

# Refit on ALL years for the real forecast, then forecast from the latest complete month.
all_train = usable
final_model = fit_model(all_train) if model_ok else None
latest = df.dropna(subset=FEATURES).iloc[[-1]]
origin = latest.index[0]
if model_ok:
    p = full_probs(final_model, latest[FEATURES])[0]
else:
    p = climatology_probs(all_train, [origin.month])[0]
p = np.round(p, 3); p[int(np.argmax(p))] = round(1 - (p.sum() - p.max()), 3)      # make the 4 numbers add to exactly 1
probs = dict(zip(KEYS, [float(x) for x in p]))
print("Forecast origin:", origin.strftime("%Y-%m"), "-> next 3 months:", probs)

## 5. Write the file the advisor reads

The advisor needs a recent date, four probabilities between 0 and 1 adding to about 1, and a Borana-specific entry per market. The file also carries a small **model card** with the test scores, so anyone can see how good (or modest) the forecast really is.

In [ ]:
today = datetime.date.today().isoformat()
card = {"method": METHOD, "data": "ERA5 rainfall via Open-Meteo, %d Borana points, %d-%s; ONI%s" % (len(POINTS), START_YEAR, END.isoformat(), " + DMI" if iod is not None else ""),
        "classes": "SPI of next 3 months: No Drought >= -0.5; Mild -1.0..-0.5; Moderate -1.5..-1.0; Severe < -1.5",
        "split": "train 1981-2012, check 2013-2018, test 2019-2025 (time-based)",
        "brier_skill_vs_climatology": {k: round(v, 3) for k, v in bss.items()}, "forecast_origin": origin.strftime("%Y-%m"),
        "limits": "rainfall only (not forage/soil moisture/deaths); modest skill expected; reanalysis not gauge data"}
outlook = {"updated": today, "source": "borana-spi-v1", "skill": ("model" if model_ok else "climatology_only"),
           "markets": {m: probs for m in MARKETS_TO_EXPORT}, "model_card": card}
with open("drought_outlook.json", "w") as f: json.dump(outlook, f, indent=1)
print(json.dumps(outlook, indent=1)[:900])
try:
    from google.colab import files
    files.download("drought_outlook.json")
except ImportError:
    print("Saved next to the notebook.")

## 6. Give it to the advisor
Either copy `drought_outlook.json` into the advisor's `data/` folder, or send it with:

    curl -X POST https://YOUR-APP/advisor/api/drought -H "X-Api-Key: YOUR_ADMIN_KEY" -H "Content-Type: application/json" -d @drought_outlook.json

The advisor ignores any forecast older than 14 days, so re-run this notebook about every 2 weeks (or monthly, when new rainfall data arrives).

**Next improvements:** use CHIRPS (gauge-blended) instead of ERA5 rainfall; add NDVI/soil moisture so the target is forage, not just rain; test seasonal ENSO/IOD forecasts as inputs; report skill separately for the long rains (Mar-May) and short rains (Oct-Dec).